### BERT : masked language modelling
*The animal did not cross the road because it was too [MASK]*

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
model.eval()

C:\Users\Dell\OneDrive\Documents\TomDocuments\PythonProjects\DL\20261001\DeepLearning\DL_MATH70116_2026\dl_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 678.76it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [2]:
text = "The animal did not cross the road because it was too [MASK]."
inputs = tokenizer(text, return_tensors="pt")

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("Tokens:", tokens)

Tokens: ['[CLS]', 'the', 'animal', 'did', 'not', 'cross', 'the', 'road', 'because', 'it', 'was', 'too', '[MASK]', '.', '[SEP]']


In [3]:
with torch.no_grad():
    logits = model(**inputs).logits 

In [4]:
print(type(logits), logits.shape)

<class 'torch.Tensor'> torch.Size([1, 15, 30522])


In [5]:
# Find the position of [MASK]
mask_pos = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0].item()
print("Mask position:", mask_pos)

Mask position: 12


In [6]:
probs = F.softmax(logits[0, mask_pos], dim=-1)
top = torch.topk(probs, 15)

print(f"\n{'rank':>4}  {'token':<12} {'prob':>8}")
for rank, (p, i) in enumerate(zip(top.values, top.indices), start=1):
    print(f"{rank:>4}  {tokenizer.convert_ids_to_tokens(int(i)):<12} {p.item():>8.4f}")


rank  token            prob
   1  fast           0.0841
   2  late           0.0828
   3  slow           0.0814
   4  dangerous      0.0735
   5  heavy          0.0606
   6  close          0.0584
   7  far            0.0422
   8  small          0.0365
   9  cold           0.0321
  10  dark           0.0277
  11  high           0.0260
  12  hot            0.0203
  13  low            0.0169
  14  large          0.0168
  15  steep          0.0165


<b>Exercise</b> Redo this for: *The man [MASK] to the shop.*

Lowest ranked tokens:

In [7]:
top = torch.topk(probs, 15, largest=False)

print(f"\n{'rank':>4}  {'token':<12} {'prob':>8}")
for rank, (p, i) in enumerate(zip(top.values, top.indices), start=1):
    print(f"{rank:>4}  {tokenizer.convert_ids_to_tokens(int(i)):<12} {p.item():>8.4f}")


rank  token            prob
   1  ##arium        0.0000
   2  ##xon          0.0000
   3  ##rator        0.0000
   4  ##ttes         0.0000
   5  ##encies       0.0000
   6  nhs            0.0000
   7  ##oper         0.0000
   8  ##eley         0.0000
   9  ##udge         0.0000
  10  wonders        0.0000
  11  ##zon          0.0000
  12  ##acle         0.0000
  13  ##isen         0.0000
  14  ##core         0.0000
  15  ##uation       0.0000
